<a href="https://colab.research.google.com/github/xEzIxX/AI-Class/blob/master/week12/spam_mail_embedding_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 배열 처리를 위한 라이브러리
import numpy as np

# 신경망 모델과 레이어 관련 라이브러리
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense

# 텍스트 전처리 관련 라이브러리
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# 학습에 사용할 이메일 제목 또는 짧은 문장 데이터
docs = [
    'additional income',
    'best price',
    'big bucks',
    'cash bonus',
    'earn extra cash',
    'spring savings certificate',
    'valero gas marketing',
    'all domestic employees',
    'nominations for oct',
    'confirmation from spinner'
]

# 정답 라벨
# 1 = 스팸 메일, 0 = 정상 메일
labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

In [3]:
# 사용할 어휘 사전 크기
# 단어들을 0~49 사이의 정수로 변환
vocab_size = 50

# 각 문장을 정수 리스트로 변환
encoded_docs = [one_hot(d, vocab_size) for d in docs]

# 인코딩 결과 확인
print(encoded_docs)

[[15, 2], [14, 19], [23, 15], [20, 42], [21, 5, 20], [16, 27, 20], [34, 45, 3], [14, 43, 21], [36, 1, 20], [30, 21, 2]]


In [4]:
# 모든 문장의 길이를 4로 통일
max_length = 4

# 길이가 짧은 문장은 뒤쪽에 0을 붙임
padded_docs = pad_sequences(
    encoded_docs,
    maxlen=max_length,
    padding='post'
)

# 패딩 결과 확인
print(padded_docs)

[[15  2  0  0]
 [14 19  0  0]
 [23 15  0  0]
 [20 42  0  0]
 [21  5 20  0]
 [16 27 20  0]
 [34 45  3  0]
 [14 43 21  0]
 [36  1 20  0]
 [30 21  2  0]]


In [5]:
# 순차적으로 레이어를 쌓는 모델 생성
model = Sequential()

# Embedding 레이어
# 정수로 된 단어 번호를 8차원 벡터로 변환
model.add(Embedding(vocab_size, 8, input_length=max_length))

# Embedding 결과를 Dense 레이어에 넣기 위해 1차원으로 펼침
model.add(Flatten())

# 출력층
# sigmoid를 사용하여 스팸일 확률을 0~1 사이 값으로 출력
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [6]:
# 모델 학습 방식 설정
# 이진 분류 문제이므로 binary_crossentropy 사용
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [7]:
# 모델 학습
# epochs=50: 전체 데이터를 50번 반복 학습
# verbose=0: 학습 과정 출력 생략
model.fit(
    padded_docs,
    labels,
    epochs=50,
    verbose=0
)

In [8]:
# 학습 데이터로 모델 성능 평가
loss, accuracy = model.evaluate(
    padded_docs,
    labels,
    verbose=0
)

# 정확도 출력
print('정확도=', accuracy)

정확도= 1.0


In [9]:
# 새롭게 테스트할 문장
test_doc = ['big income']

# 테스트 문장도 학습 데이터와 같은 방식으로 정수 인코딩
encoded_test = [one_hot(d, vocab_size) for d in test_doc]

# 테스트 문장도 길이를 4로 맞춤
padded_test = pad_sequences(
    encoded_test,
    maxlen=max_length,
    padding='post'
)

# 스팸일 확률 예측
prediction = model.predict(padded_test)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
[[0.58058965]]


In [10]:
# 예측값이 0.5보다 크면 스팸으로 판단
if prediction[0][0] > 0.5:
    print("스팸 메일입니다.")
else:
    print("정상 메일입니다.")

print("스팸일 확률:", prediction[0][0])

스팸 메일입니다.
스팸일 확률: 0.58058965
